# 复现实验：ComfyUI + Cloudflare Tunnel

从零开始：clone ComfyUI → 安装依赖（含本地 ROCm PyTorch wheel）→ 启动服务 → cloudflared 隧道 → 公网访问。

## 1. 克隆 ComfyUI

In [ ]:
import os, subprocess

COMFYUI_DIR = '/workspace/ComfyUI'
if not os.path.exists(COMFYUI_DIR):
    subprocess.run(['git', 'clone',
        'https://github.com/comfyanonymous/ComfyUI.git',
        COMFYUI_DIR])
    print('克隆完成')
else:
    print('ComfyUI 已存在，跳过')

## 2. 创建虚拟环境并安装依赖

venv 创建在 `/workspace/venv`，然后从本地 wheel 安装 ROCm PyTorch。

In [ ]:
VENV_DIR = '/workspace/venv'
PYPI_MIRROR = 'https://pypi.tuna.tsinghua.edu.cn/simple'

if not os.path.exists(VENV_DIR):
    subprocess.run(['python3', '-m', 'venv', VENV_DIR])
    print('venv 已创建')

PIP = os.path.join(VENV_DIR, 'bin', 'pip')

In [ ]:
# 安装 requirements.txt（使用 PyPI 镜像）
subprocess.run([PIP, 'install', '-r',
    os.path.join(COMFYUI_DIR, 'requirements.txt'),
    '-i', PYPI_MIRROR])
print('Core 依赖安装完成')

In [ ]:
# 安装本地 ROCm PyTorch wheel（--no-deps --force-reinstall 覆盖 pip 自动安装的版本）
wheels = [
    '/torch-2.9.1+rocm7.2.1.lw.gitff65f5bc-cp312-cp312-linux_x86_64.whl',
    '/torchvision-0.24.0+rocm7.2.1.gitb919bd0c-cp312-cp312-linux_x86_64.whl',
    '/torchaudio-2.9.0+rocm7.2.1.gite3c6ee2b-cp312-cp312-linux_x86_64.whl',
    '/triton-3.5.1+rocm7.2.1.gita272dfa8-cp312-cp312-linux_x86_64.whl',
]
for w in wheels:
    if os.path.exists(w):
        subprocess.run([PIP, 'install', w, '--no-deps', '--force-reinstall'])
        print(f'已安装: {os.path.basename(w)}')
    else:
        print(f'跳过（文件不存在）: {os.path.basename(w)}')

# 安装 torchsde（无依赖）
subprocess.run([PIP, 'install', 'torchsde', '--no-deps', '-i', PYPI_MIRROR])
print('torchsde 安装完成')

## 3. 修改 requirements.txt（可选）

注释掉 torch 行，防止后续 `pip install -r requirements.txt` 时重新下载公共版 PyTorch 覆盖本地 ROCm 版本。

In [ ]:
req_path = os.path.join(COMFYUI_DIR, 'requirements.txt')
with open(req_path) as f:
    content = f.read()

for pkg in ['torch', 'torchsde', 'torchvision', 'torchaudio']:
    content = content.replace(f'{pkg}\n', f'# {pkg}\n')

with open(req_path, 'w') as f:
    f.write(content)
print('requirements.txt 已更新')

## 4. 安装 cloudflared

In [ ]:
if subprocess.run(['which', 'cloudflared'], capture_output=True).returncode != 0:
    subprocess.run(['mkdir', '-p', '--mode=0755', '/usr/share/keyrings'], capture_output=True)
    subprocess.run(['curl', '-fsSL', 'https://pkg.cloudflare.com/cloudflare-main.gpg',
                    '-o', '/usr/share/keyrings/cloudflare-main.gpg'], capture_output=True)
    with open('/etc/apt/sources.list.d/cloudflared.list', 'w') as f:
        f.write('deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main\n')
    subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'cloudflared'])
    print('cloudflared 安装完成')
else:
    print('cloudflared 已安装')

## 5. 启动 ComfyUI

In [ ]:
import time
subprocess.run(['pkill', '-9', '-f', 'ComfyUI/main.py'], capture_output=True)
time.sleep(1)

cmd = [
    os.path.join(VENV_DIR, 'bin', 'python'),
    os.path.join(COMFYUI_DIR, 'main.py'),
    '--port', '8188',
    '--listen', '127.0.0.1',
    '--enable-compress-response-body',
    '--enable-cors-header', '*'
]
with open('/tmp/comfyui.log', 'w') as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=log)
print(f'ComfyUI PID: {proc.pid}')

for i in range(30):
    r = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}',
                       'http://127.0.0.1:8188/'], capture_output=True, text=True, timeout=3)
    if r.stdout == '200':
        print('ComfyUI 就绪')
        break
    time.sleep(1)
else:
    print('超时，请检查 /tmp/comfyui.log')

## 6. 启动 Cloudflare Tunnel

In [ ]:
import re
subprocess.run(['pkill', '-9', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(1)

cmd = ['cloudflared', 'tunnel', '--protocol', 'http2', '--url', 'http://127.0.0.1:8188']
with open('/tmp/cloudflared.log', 'w') as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=log)
print(f'cloudflared PID: {proc.pid}')

url_pat = re.compile(r'https://[a-z-]+\.trycloudflare\.com')
tunnel_url = None
for i in range(30):
    time.sleep(1)
    try:
        with open('/tmp/cloudflared.log') as f:
            m = url_pat.search(f.read())
            if m:
                tunnel_url = m.group(0)
                break
    except FileNotFoundError:
        pass

if tunnel_url:
    print(f'隧道地址: {tunnel_url}')
else:
    print('超时，请检查 /tmp/cloudflared.log')

## 7. 验证

In [ ]:
if tunnel_url:
    r = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}',
                       tunnel_url + '/'], capture_output=True, text=True, timeout=10)
    print(f'隧道状态: HTTP {r.stdout}')
    if r.stdout == '200':
        print(f'成功！浏览器打开: {tunnel_url}')

## 8. 停止服务

In [ ]:
subprocess.run(['pkill', '-9', '-f', 'cloudflared tunnel'], capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'ComfyUI/main.py'], capture_output=True)
print('已停止')